In [1]:
import pandas as pd
import numpy as np

# Load the original merged dataset
df_base = pd.read_csv("abide_merged.csv")

print("=" * 60)
print("BASE DATASET LOADED: abide_merged.csv")
print("=" * 60)
print(f"Shape: {df_base.shape[0]} rows × {df_base.shape[1]} columns")
print(f"ASD:     {(df_base['DX_LABEL']=='ASD').sum()}")
print(f"Control: {(df_base['DX_LABEL']=='Control').sum()}")
print(f"Sites:   {df_base['SITE_ID'].nunique()}")

BASE DATASET LOADED: abide_merged.csv
Shape: 1035 rows × 81 columns
ASD:     505
Control: 530
Sites:   20


onvert sentinel values (-9999) to NaN

In [2]:
# Convert known sentinel codes (-9999) to real NaN
# This applies to numeric phenotypic columns AND the string HANDEDNESS_CATEGORY column
df_base[["FIQ", "VIQ", "PIQ", "ADOS_TOTAL"]] = df_base[["FIQ", "VIQ", "PIQ", "ADOS_TOTAL"]].replace(-9999, np.nan)

# HANDEDNESS_CATEGORY has '-9999' as a STRING — needs separate handling
df_base["HANDEDNESS_CATEGORY"] = df_base["HANDEDNESS_CATEGORY"].replace("-9999", np.nan)

# Identify thickness columns once (used throughout)
THICKNESS_COLS = [c for c in df_base.columns if c.startswith(("lh_", "rh_"))]

print("=" * 60)
print("STEP: Sentinel values (-9999) converted to NaN")
print("=" * 60)
print(f"Shape: {df_base.shape[0]} rows × {df_base.shape[1]} columns")
print(f"Total NaN count: {df_base.isnull().sum().sum()}")
print(f"Thickness columns identified: {len(THICKNESS_COLS)}")

STEP: Sentinel values (-9999) converted to NaN
Shape: 1035 rows × 81 columns
Total NaN count: 1379
Thickness columns identified: 68


Drop subjects with missing thickness values 

In [3]:
# Subjects with any missing thickness value have likely unreliable segmentations
# Drop them entirely (applies to both Solution A and Solution B)
n_before = len(df_base)
df_base = df_base.dropna(subset=THICKNESS_COLS).reset_index(drop=True)
n_after = len(df_base)

print("=" * 60)
print("STEP: Drop subjects with missing thickness values")
print("=" * 60)
print(f"Subjects removed: {n_before - n_after}")
print(f"Shape: {df_base.shape[0]} rows × {df_base.shape[1]} columns")
print(f"ASD:     {(df_base['DX_LABEL']=='ASD').sum()}")
print(f"Control: {(df_base['DX_LABEL']=='Control').sum()}")

STEP: Drop subjects with missing thickness values
Subjects removed: 8
Shape: 1027 rows × 81 columns
ASD:     499
Control: 528


 Drop subjects with biologically impossible thickness values (<1mm or >5mm)



In [4]:
# Apply biological plausibility filter — cortical thickness outside [1.0, 5.0] mm is impossible
# Indicates FreeSurfer failure, drop the whole subject
bio_mask = ((df_base[THICKNESS_COLS] < 1.0) | (df_base[THICKNESS_COLS] > 5.0)).any(axis=1)
n_removed_bio = bio_mask.sum()

df_base = df_base[~bio_mask].reset_index(drop=True)

print("=" * 60)
print("STEP: Drop subjects with biologically impossible thickness (<1mm or >5mm)")
print("=" * 60)
print(f"Subjects removed: {n_removed_bio}")
print(f"Shape: {df_base.shape[0]} rows × {df_base.shape[1]} columns")
print(f"ASD:     {(df_base['DX_LABEL']=='ASD').sum()}")
print(f"Control: {(df_base['DX_LABEL']=='Control').sum()}")

STEP: Drop subjects with biologically impossible thickness (<1mm or >5mm)
Subjects removed: 0
Shape: 1027 rows × 81 columns
ASD:     499
Control: 528


## SOL A — Site-median with global fallback

Create df_A and impute FIQ/VIQ/PIQ with site-median + global fallback

In [5]:
# Solution A: keep FIQ/VIQ/PIQ columns, impute with site median, 
# fall back to global median if a site has no data for that variable
df_A = df_base.copy()

IQ_COLS = ["FIQ", "VIQ", "PIQ"]

print("=" * 60)
print("SOLUTION A — Site-median imputation (with global fallback)")
print("=" * 60)

for col in IQ_COLS:
    n_missing_before = df_A[col].isnull().sum()
    
    # Step 1: fill with site-specific median (may still leave NaN if the site has zero data)
    df_A[col] = df_A.groupby("SITE_ID")[col].transform(lambda x: x.fillna(x.median()))
    n_after_site = df_A[col].isnull().sum()
    
    # Step 2: fallback — fill remaining NaN with global median
    global_med = df_A[col].median()
    df_A[col] = df_A[col].fillna(global_med)
    n_after_global = df_A[col].isnull().sum()
    
    print(f"\n{col}:")
    print(f"  Missing before:                 {n_missing_before}")
    print(f"  Filled by site median:          {n_missing_before - n_after_site}")
    print(f"  Filled by global median ({global_med:.1f}):   {n_after_site - n_after_global}")
    print(f"  Missing after:                  {n_after_global}")

print(f"\ndf_A shape: {df_A.shape[0]} rows × {df_A.shape[1]} columns")

SOLUTION A — Site-median imputation (with global fallback)

FIQ:
  Missing before:                 69
  Filled by site median:          35
  Filled by global median (109.0):   34
  Missing after:                  0

VIQ:
  Missing before:                 179
  Filled by site median:          19
  Filled by global median (108.0):   160
  Missing after:                  0

PIQ:
  Missing before:                 165
  Filled by site median:          57
  Filled by global median (108.0):   108
  Missing after:                  0

df_A shape: 1027 rows × 81 columns


 Handle HANDEDNESS_CATEGORY (convert NaN to "Unknown")

In [6]:
# Treat missing handedness as its own category rather than dropping information
n_missing = df_A["HANDEDNESS_CATEGORY"].isnull().sum()
df_A["HANDEDNESS_CATEGORY"] = df_A["HANDEDNESS_CATEGORY"].fillna("Unknown")

print("=" * 60)
print("SOLUTION A — HANDEDNESS_CATEGORY: NaN → 'Unknown'")
print("=" * 60)
print(f"Converted to 'Unknown': {n_missing}")
print(f"\nValue counts:")
print(df_A["HANDEDNESS_CATEGORY"].value_counts())
print(f"\ndf_A shape: {df_A.shape[0]} rows × {df_A.shape[1]} columns")

SOLUTION A — HANDEDNESS_CATEGORY: NaN → 'Unknown'
Converted to 'Unknown': 286

Value counts:
HANDEDNESS_CATEGORY
R          655
Unknown    286
L           66
Ambi        13
Mixed        6
L->R         1
Name: count, dtype: int64

df_A shape: 1027 rows × 81 columns


ADOS_TOTAL stays untouched (MNAR, do not impute)


In [7]:
# ADOS is administered only to ASD subjects by design → do not impute
# Keep the column as-is (NaN in Control is correct)
print("=" * 60)
print("SOLUTION A — ADOS_TOTAL: NOT imputed (MNAR by design)")
print("=" * 60)
print(f"ADOS_TOTAL missing in ASD:     {df_A[df_A['DX_LABEL']=='ASD']['ADOS_TOTAL'].isnull().sum()} / {(df_A['DX_LABEL']=='ASD').sum()}")
print(f"ADOS_TOTAL missing in Control: {df_A[df_A['DX_LABEL']=='Control']['ADOS_TOTAL'].isnull().sum()} / {(df_A['DX_LABEL']=='Control').sum()}")
print(f"\ndf_A shape: {df_A.shape[0]} rows × {df_A.shape[1]} columns")

SOLUTION A — ADOS_TOTAL: NOT imputed (MNAR by design)
ADOS_TOTAL missing in ASD:     149 / 499
ADOS_TOTAL missing in Control: 508 / 528

df_A shape: 1027 rows × 81 columns


In [8]:
print("=" * 60)
print("SOLUTION A — FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df_A.shape[0]} rows × {df_A.shape[1]} columns")
print(f"ASD:     {(df_A['DX_LABEL']=='ASD').sum()}")
print(f"Control: {(df_A['DX_LABEL']=='Control').sum()}")
print(f"Sites:   {df_A['SITE_ID'].nunique()}")
print(f"\nMissing values per column (only columns with any missing):")
miss_A = df_A.isnull().sum()
print(miss_A[miss_A > 0].to_string())

# Save for later use
df_A.to_csv("abide_solution_A.csv", index=False)
print("\nSaved: abide_solution_A.csv")

SOLUTION A — FINAL DATASET SUMMARY
Shape: 1027 rows × 81 columns
ASD:     499
Control: 528
Sites:   20

Missing values per column (only columns with any missing):
ADOS_TOTAL    657

Saved: abide_solution_A.csv


## Drop problematic columns: FIQ imputation

Create df_B and drop VIQ, PIQ, HANDEDNESS_CATEGORY columns

In [9]:
# Solution B: drop columns that have high site-structured missingness
# VIQ, PIQ are entirely missing from several sites → imputing them would fabricate data
# HANDEDNESS_CATEGORY is entirely missing from NYU and USM
df_B = df_base.copy()

cols_to_drop = ["VIQ", "PIQ", "HANDEDNESS_CATEGORY"]
df_B = df_B.drop(columns=cols_to_drop)

print("=" * 60)
print("SOLUTION B — Drop high-missingness columns")
print("=" * 60)
print(f"Dropped columns: {cols_to_drop}")
print(f"Shape: {df_B.shape[0]} rows × {df_B.shape[1]} columns")

SOLUTION B — Drop high-missingness columns
Dropped columns: ['VIQ', 'PIQ', 'HANDEDNESS_CATEGORY']
Shape: 1027 rows × 78 columns


Impute FIQ with site-median + global fallback

In [10]:
# FIQ has only 6.7% missing and is not site-structurally absent (every site has some data)
# Safe to impute with site-median, with global-median fallback for safety
col = "FIQ"
n_missing_before = df_B[col].isnull().sum()

# Site-level median imputation
df_B[col] = df_B.groupby("SITE_ID")[col].transform(lambda x: x.fillna(x.median()))
n_after_site = df_B[col].isnull().sum()

# Fallback: global median
global_med = df_B[col].median()
df_B[col] = df_B[col].fillna(global_med)
n_after_global = df_B[col].isnull().sum()

print("=" * 60)
print("SOLUTION B — FIQ site-median imputation (global fallback)")
print("=" * 60)
print(f"Missing before:                 {n_missing_before}")
print(f"Filled by site median:          {n_missing_before - n_after_site}")
print(f"Filled by global median ({global_med:.1f}):   {n_after_site - n_after_global}")
print(f"Missing after:                  {n_after_global}")
print(f"\ndf_B shape: {df_B.shape[0]} rows × {df_B.shape[1]} columns")

SOLUTION B — FIQ site-median imputation (global fallback)
Missing before:                 69
Filled by site median:          35
Filled by global median (109.0):   34
Missing after:                  0

df_B shape: 1027 rows × 78 columns


ADOS_TOTAL stays untouched (MNAR, do not impute)

In [11]:
# Same logic as Solution A: ADOS is MNAR by design, keep NaN for Control subjects
print("=" * 60)
print("SOLUTION B — ADOS_TOTAL: NOT imputed (MNAR by design)")
print("=" * 60)
print(f"ADOS_TOTAL missing in ASD:     {df_B[df_B['DX_LABEL']=='ASD']['ADOS_TOTAL'].isnull().sum()} / {(df_B['DX_LABEL']=='ASD').sum()}")
print(f"ADOS_TOTAL missing in Control: {df_B[df_B['DX_LABEL']=='Control']['ADOS_TOTAL'].isnull().sum()} / {(df_B['DX_LABEL']=='Control').sum()}")
print(f"\ndf_B shape: {df_B.shape[0]} rows × {df_B.shape[1]} columns")

SOLUTION B — ADOS_TOTAL: NOT imputed (MNAR by design)
ADOS_TOTAL missing in ASD:     149 / 499
ADOS_TOTAL missing in Control: 508 / 528

df_B shape: 1027 rows × 78 columns


 summary for df_B

In [13]:
print("=" * 60)
print("SOLUTION B — FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df_B.shape[0]} rows × {df_B.shape[1]} columns")
print(f"ASD:     {(df_B['DX_LABEL']=='ASD').sum()}")
print(f"Control: {(df_B['DX_LABEL']=='Control').sum()}")
print(f"Sites:   {df_B['SITE_ID'].nunique()}")
print(f"\nMissing values per column (only columns with any missing):")
miss_B = df_B.isnull().sum()
print(miss_B[miss_B > 0].to_string())

# Save for later use
df_B.to_csv("abide_solution_B.csv", index=False)
print("\nSaved: abide_solution_B.csv")

SOLUTION B — FINAL DATASET SUMMARY
Shape: 1027 rows × 78 columns
ASD:     499
Control: 528
Sites:   20

Missing values per column (only columns with any missing):
ADOS_TOTAL    657

Saved: abide_solution_B.csv


compare

In [14]:
print("=" * 60)
print("FINAL COMPARISON — df_A vs df_B")
print("=" * 60)

comparison = pd.DataFrame({
    "df_A (site-median)":  [df_A.shape[0], df_A.shape[1], 
                            (df_A['DX_LABEL']=='ASD').sum(),
                            (df_A['DX_LABEL']=='Control').sum(),
                            df_A.isnull().sum().sum()],
    "df_B (drop columns)": [df_B.shape[0], df_B.shape[1],
                            (df_B['DX_LABEL']=='ASD').sum(),
                            (df_B['DX_LABEL']=='Control').sum(),
                            df_B.isnull().sum().sum()]
}, index=["Rows (subjects)", "Columns", "ASD count", "Control count", "Total NaN"])

print(comparison.to_string())

print("\n--- What's in each dataset ---")
print(f"df_A has these phenotypic columns: FIQ, VIQ, PIQ, ADOS_TOTAL, HANDEDNESS_CATEGORY")
print(f"df_B has these phenotypic columns: FIQ, ADOS_TOTAL")
print(f"\nBoth keep all 68 thickness columns.")
print(f"Both preserve ADOS_TOTAL as NaN (MNAR).")

FINAL COMPARISON — df_A vs df_B
                 df_A (site-median)  df_B (drop columns)
Rows (subjects)                1027                 1027
Columns                          81                   78
ASD count                       499                  499
Control count                   528                  528
Total NaN                       657                  657

--- What's in each dataset ---
df_A has these phenotypic columns: FIQ, VIQ, PIQ, ADOS_TOTAL, HANDEDNESS_CATEGORY
df_B has these phenotypic columns: FIQ, ADOS_TOTAL

Both keep all 68 thickness columns.
Both preserve ADOS_TOTAL as NaN (MNAR).
